IMPORTS

In [3]:
import pandas as pd 
import numpy as np

LOAD DATA

In [4]:
df = pd.read_csv("../data/processed/high_risk_customers.csv")

print("Shape:", df.shape)
df.head()

Shape: (1726, 26)


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,...,PaymentMethod,MonthlyCharges,TotalCharges,Churn,tenure_group,num_services,contract_risk,engagement_score,Churn_probability,cluster
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,...,Electronic check,29.85,29.85,No,New,1,2,1.381833,0.830421,0
1,Female,0,No,No,2,Yes,No,Fiber optic,No,No,...,Electronic check,70.70,151.65,Yes,New,1,2,1.873667,0.831514,0
2,Female,0,No,No,8,Yes,Yes,Fiber optic,No,No,...,Electronic check,99.65,820.50,Yes,New,5,2,6.663167,0.958987,1
3,Female,0,Yes,No,28,Yes,Yes,Fiber optic,No,No,...,Electronic check,104.80,3046.05,Yes,Mid,6,2,9.381333,0.834971,2
4,Male,0,No,No,49,Yes,Yes,Fiber optic,No,Yes,...,Bank transfer (automatic),103.70,5036.30,Yes,Loyal,6,2,11.120333,0.778060,2


CREATE A/B GROUPS

In [5]:
np.random.seed(42)

df['group'] = np.random.choice(['control','treatment'], size=len(df))

df['group'].value_counts()

group
control      873
treatment    853
Name: count, dtype: int64

SIMULATE CHURN (CONTROL GROUP)

In [6]:
df["Churn_simulated"] = df['Churn_probability']

APPLY TREATMENT EFFECT

Assume campaign reduces churn by 25%

In [7]:
treatment_effect = 0.25

df.loc[df['group'] == 'treatment','Churn_simulated'] = \
df['Churn_probability'] * (1 - treatment_effect)

COMPARE CHURN RATES

In [8]:
ab_result = df.groupby('group')['Churn_simulated'].mean()
ab_result

group
control      0.836123
treatment    0.627465
Name: Churn_simulated, dtype: float64

CALCULATE LIFT

In [10]:
control_rate = ab_result['control']
treatment_rate = ab_result['treatment']

lift = control_rate - treatment_rate

print("Churn Reduction (Lift):", lift)

Churn Reduction (Lift): 0.20865807514332202


REVENUE IMPACT FROM A/B TEST

In [11]:
df['Monthly_revenue'] = df['MonthlyCharges']

#Expected loss without treatment
control_loss = df[df['group'] == 'control']['Monthly_revenue'].sum()

#Expected loss with treatment (reduce churn)
treatment_loss = df[df['group'] == 'treatment']['Monthly_revenue'].sum() * (1 - treatment_effect)

revenue_saved = control_loss - treatment_loss

print("Estimated revenue saved via A/B test (₹):", revenue_saved)

Estimated revenue saved via A/B test (₹): 18717.749999999993
